# 03 - SQL Injections

This notebook explores another feature of **RTGL** which in some cases can handle more complex *multi-hop relationships* (where there is no single shortest temporal path between tables) within **Relational Deep Learning**. 

Unlike standard tasks or **CPE**s, **SQL Injections** allow the user to explicitly define the required path by injecting custom SQL. Most importantly, it unlocks the full power of SQL, enabling the use of advanced database functions and operations that are not yet natively supported in **RTGL**.

The [**RelBench**](https://relbench.stanford.edu/) framework is used as the primary source of data and tasks, leveraging its collection of pre-defined tasks to evaluate **RTGL**'s capabilities.

## Table of Contents
1. [F1 Dataset](#f1-dataset)
    - 1.1 [Link Prediction Tasks](#f1-link-tasks)
        - 1.1.1 [driver-circuit-compete](#driver-circuit-compete)
2. [Stack-Exchange Q&A Website Dataset](#stack-exchange-dataset)
    - 2.1 [Link Prediction Tasks](#stack-link-tasks)
        - 2.1.1 [post-post-related](#post-post-related)
3. [Amazon e-commerce Dataset](#amazon-dataset)
    - 3.1 [Entity Regression Tasks](#amazon-reg-tasks)
        - 3.1.1 [item-ltv](#item-ltv)
        - 3.1.2 [user-item-review](#user-item-review)
4. [arXiv Dataset](#arxiv-dataset)
    - 4.1 [Entity Classification Tasks](#arxiv-clas-tasks)
        - 4.1.1 [paper-citation](#paper-citation)

In [29]:
%load_ext autoreload
%autoreload 2

from experiments.utils import load_dataset_rb, load_task_rb, check_correctness

## 1. F1 Dataset <a id="f1-dataset"></a>

In this section, we attempt to generate the same tasks from the `F1 Dataset` which are already pre-defined in *RelBench*.

In [30]:
dataset_f1 = load_dataset_rb(name="rel-f1")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

### 1.1 Link Prediction Tasks <a id="f1-link-tasks"></a>

#### 1.1.1 driver-circuit-compete <a id="driver-circuit-compete"></a>

Task Description: Predict on which circuits a driver will compete in the next 1 year.


In [31]:
task_f1_driver_circuit = load_task_rb(dataset_f1, "driver-circuit-compete")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [32]:
rtgl_query = """
    PREDICT LIST_DISTINCT(
        [
            SELECT 
                dr.driverId,
                rac.circuitId,
                rac.date
            FROM
                drivers as dr
            JOIN 
                results as res
            ON 
                res.driverId = dr.driverId
            JOIN
                races as rac
            ON
                rac.raceId = res.raceId
        ]{circuits_drivers}
        {}
        {driverId->drivers, circuitId->circuits}
        {}
        {date}.circuitId, 0, 365, DAYS)
    FOR EACH drivers.*;
"""

In [33]:
# TRAIN

check_correctness(dataset_f1, task_f1_driver_circuit, rtgl_query, split="train")

TIMEDELTA: 365 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'circuits_drivers' has multiple temporal paths to parent table 'drivers'. Using the shortest one: circuits_drivers.driverid -> drivers.driverid


SQL query executed in 0.09 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'driverId': 'drivers', 'circuitId': 'circuits'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers', 'label': 'circuits'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       timestamp   fk                                              label _merge
0    2000-01-03    1  (0, 1, 3, 5, 6, 7, 8, 9, 10, 12, 13, 17, 18, 2...   both
1    2001-01-02    1  (0, 1, 3, 5, 6, 7, 8, 9, 10, 12, 13, 17, 18, 1...   both
2    2002-01-02    1  (0, 1, 3, 5, 6, 7, 8, 9, 10, 12, 13, 17, 18, 1...   both
3    2003-01-02    1  (0, 1, 3, 5, 6, 7, 8, 9, 10, 13, 17, 18, 19, 2...   both
4    2004-01-02    1  (0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 12, 13, 16, 17...   both
...         ...  ...                                          

In [34]:
# VAL

check_correctness(dataset_f1, task_f1_driver_circuit, rtgl_query, split="val")

TIMEDELTA: 365 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'circuits_drivers' has multiple temporal paths to parent table 'drivers'. Using the shortest one: circuits_drivers.driverid -> drivers.driverid


SQL query executed in 0.03 seconds
------------------- START VAL -------------------
RelBench fkeys: {'driverId': 'drivers', 'circuitId': 'circuits'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers', 'label': 'circuits'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
     timestamp  fk                                              label _merge
0  2005-01-01   1     (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 18, 19, 20)   both
1  2005-01-01   3  (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 16,...   both
2  2005-01-01   7  (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 16,...   both
3  2005-01-01  10  (0, 2, 4, 6, 7, 8, 9, 10, 12, 13, 16, 17, 18, ...   both
4  2005-01-01  12  (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 16,...   both
5  2005-01-01  13  (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 16,...   both
6  200

In [35]:
# TEST

check_correctness(dataset_f1, task_f1_driver_circuit, rtgl_query, split="test")

TIMEDELTA: 365 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'circuits_drivers' has multiple temporal paths to parent table 'drivers'. Using the shortest one: circuits_drivers.driverid -> drivers.driverid


SQL query executed in 0.03 seconds
------------------- START TEST -------------------
RelBench fkeys: {'driverId': 'drivers', 'circuitId': 'circuits'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers', 'label': 'circuits'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
     timestamp   fk                                              label _merge
0  2010-01-01    0  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   both
1  2010-01-01    1                               (14, 17, 21, 23, 34)   both
2  2010-01-01    2  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   both
3  2010-01-01    3  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   both
4  2010-01-01    4  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   both
5  2010-01-01    8  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   bot

## 2. Stack-Exchange Q&A Website Dataset <a id="stack-exchange-dataset"></a>

In this section, we attempt to generate the same tasks from the `Stack-Exchange Q&A Website Dataset` which are already pre-defined in *RelBench*.


In [36]:
dataset_stack = load_dataset_rb(name="rel-stack")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

### 2.1 Link Prediction Tasks <a id="stack-link-tasks"></a>


#### 2.1.1 post-post-related <a id="post-post-related"></a>

Task Description: Predict a list of existing posts that users will link a given post to in the next two months.

**Note:** This task is difficult to define without a **CPE**, and even using **SQL Injections** proves to be highly problematic. 
The core issue in both cases remains the same: there are two distinct columns pointing to the `posts` table, creating unavoidable ambiguity that **CPE**s are specifically designed to resolve.

## 3. Amazon e-commerce Dataset <a id="amazon-dataset"></a>

In this section, we attempt to generate the same tasks from the `Amazon e-commerce Dataset` which are already pre-defined in *RelBench*.


In [37]:
dataset_amazon = load_dataset_rb(name="rel-amazon")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]

### 3.1 Entity Regression Tasks <a id="amazon-reg-tasks"></a>


#### 3.1.1 item-ltv <a id="item-ltv"></a>

Task Description: For each product, predict the $ value of the total number purchases and reviews it recieves in the next 3 months.


In [38]:
task_amazon_item_ltv = load_task_rb(dataset_amazon, "item-ltv")

In [39]:
rtgl_query = """
     PREDICT SUM(
          [
               SELECT
                    p.product_id,
                    p.price,
                    r.review_time
               FROM
                    product as p
               JOIN
                    review as r
               ON
                   r.product_id = p.product_id
          ]{product_product}
          {}
          {product_id->product}
          {}
          {review_time}.price, 0, 91, DAYS)
     FOR EACH product.*
     ASSUMING COUNT(review.*, 0, 91, DAYS) != 0;
"""

In [40]:
# TRAIN

check_correctness(dataset_amazon, task_amazon_item_ltv, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 9.47 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk    label _merge
0       2012-10-04       0   899.99   both
1       2013-01-03       0  2197.65   both
2       2013-04-04       0  1737.19   both
3       2013-07-04       0  2030.21   both
4       2013-10-03       0  1862.77   both
...            ...     ...      ...    ...
2707674 2013-10-03  506009    85.86   both
2707675 2014-07-03  506009   171.72   both
2707676 2014-10-02  506009   171.72   both
2707677 2008-10-09  506010    17.87   both
2707678 2012-10-04  506010    17.87   both

[2707679 rows x 4 columns]
------------------- END TRAIN -

In [41]:
# VAL

check_correctness(dataset_amazon, task_amazon_item_ltv, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.42 seconds
------------------- START VAL -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk    label _merge
0      2015-10-01       0  2344.16   both
1      2015-10-01       3   159.95   both
2      2015-10-01       4   358.08   both
3      2015-10-01       5     3.99   both
4      2015-10-01       6    47.70   both
...           ...     ...      ...    ...
166973 2015-10-01  505996    21.35   both
166974 2015-10-01  505998    17.69   both
166975 2015-10-01  505999   111.18   both
166976 2015-10-01  506000  1277.92   both
166977 2015-10-01  506002    19.82   both

[166978 rows x 4 colu

In [42]:
# TEST

check_correctness(dataset_amazon, task_amazon_item_ltv, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.55 seconds
------------------- START TEST -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk    label _merge
0      2016-01-01       0  2448.81   both
1      2016-01-01       1    23.98   both
2      2016-01-01       4   134.28   both
3      2016-01-01       6    23.85   both
4      2016-01-01       7     5.78   both
...           ...     ...      ...    ...
178329 2016-01-01  505998    35.38   both
178330 2016-01-01  505999    74.12   both
178331 2016-01-01  506000  1069.28   both
178332 2016-01-01  506002    19.82   both
178333 2016-01-01  506008    17.00   both

[178334 rows x 4 col

#### 3.1.2 user-item-review <a id="user-item-review"></a>

Task Description: Predict the list of distinct items each customer will purchase and give a detailed review in the next 3 months.

Note: This task cannot be implemented using pure **RTGL** alone. Currently, the framework does not support specific SQL functions such as `LENGTH`.

In [43]:
task_amazon_user_item_review = load_task_rb(dataset_amazon, "user-item-review")

In [44]:
rtgl_query = """
     PREDICT LIST_DISTINCT([
          SELECT
               *
          FROM
               review
          WHERE 
               LENGTH(review_text) > 300
            AND
               review_text IS NOT NULL
          ]{filtered_review}
           {}
           {product_id->product, customer_id->customer}
           {}
           {review_time}.product_id, 0, 91, DAYS)
     FOR EACH customer.*;
"""

In [45]:
# TRAIN

check_correctness(dataset_amazon, task_amazon_user_item_review, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'filtered_review' has multiple temporal paths to parent table 'customer'. Using the shortest one: filtered_review.customer_id -> customer.customer_id


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 11.11 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk             label _merge
0       2008-04-10        0         (465326,)   both
1       2010-10-07        0          (93869,)   both
2       2011-04-07        0         (297923,)   both
3       2012-01-05        0         (297644,)   both
4       2012-04-05        0         (413368,)   both
...            ...      ...               ...    ...
2324172 2009-01-08  1850147         (503320,)   both
2324173 2015-01-01  1850153  (337213, 337946)   both
2324174 2015-01-01  1850154  (337213, 337946)   b

In [46]:
# VAL

check_correctness(dataset_amazon, task_amazon_user_item_review, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'filtered_review' has multiple temporal paths to parent table 'customer'. Using the shortest one: filtered_review.customer_id -> customer.customer_id


SQL query executed in 1.27 seconds
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2015-10-01        5                                     (26989, 81149)   
1      2015-10-01       23                                           (30100,)   
2      2015-10-01       25                                    (56023, 396864)   
3      2015-10-01       30                                           (39271,)   
4      2015-10-01       48  (11824, 13491, 41970, 44511, 54705, 68058, 741...   
...           ...      ...                 

In [47]:
# TEST

check_correctness(dataset_amazon, task_amazon_user_item_review, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'filtered_review' has multiple temporal paths to parent table 'customer'. Using the shortest one: filtered_review.customer_id -> customer.customer_id


SQL query executed in 1.51 seconds
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2016-01-01        0                                          (125393,)   
1      2016-01-01        5  (6311, 81593, 81748, 81751, 81800, 88501, 2040...   
2      2016-01-01       19                                               (1,)   
3      2016-01-01       20                                          (237731,)   
4      2016-01-01       25                                    (25632, 440766)   
...           ...      ...                

## 4. arXiv Dataset <a id="arxiv-dataset"></a>

In this section, we attempt to generate the same tasks from the `arXiv Dataset` which are already pre-defined in *RelBench*.

In [48]:
dataset_arxiv = load_dataset_rb(name="rel-arxiv")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

### 4.1 Entity Classification Tasks <a id="arxiv-clas-tasks"></a>

#### 4.1.1 paper-citation <a id="paper-citation"></a>

Task Description: Predict if a paper gets cited in the next 6 months.

In [49]:
task_arxiv_paper_citation = load_task_rb(dataset_arxiv, "paper-citation")

In [50]:
rtgl_query = """
    PREDICT COUNT(
        [
            SELECT
                *
            FROM
                citations
        ]{citations_papers}
        {}
        {References_Paper_ID->papers}
        {}
        {Submission_Date}.*, 0, 182, DAYS) != 0
    FOR EACH papers.*;
"""

In [51]:
# TRAIN

check_correctness(dataset_arxiv, task_arxiv_paper_citation, rtgl_query, split="train")

TIMEDELTA: 182 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.37 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'Paper_ID': 'papers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'papers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2018-01-06       0     0   both
1      2018-07-07       0     0   both
2      2019-01-05       0     0   both
3      2019-07-06       0     0   both
4      2020-01-04       0     0   both
...           ...     ...   ...    ...
534228 2021-07-03  136178     0   both
534229 2021-07-03  136179     0   both
534230 2021-07-03  136180     0   both
534231 2021-07-03  136181     0   both
534232 2021-07-03  136182     0   both

[534233 rows x 4 columns]
------------------- END TRAIN -------

In [52]:
# VAL

check_correctness(dataset_arxiv, task_arxiv_paper_citation, rtgl_query, split="val")

TIMEDELTA: 182 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.13 seconds
------------------- START VAL -------------------
RelBench fkeys: {'Paper_ID': 'papers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'papers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2022-01-01       0     0   both
1      2022-01-01       1     0   both
2      2022-01-01       2     1   both
3      2022-01-01       3     0   both
4      2022-01-01       4     0   both
...           ...     ...   ...    ...
155840 2022-01-01  155840     0   both
155841 2022-01-01  155841     0   both
155842 2022-01-01  155842     0   both
155843 2022-01-01  155843     0   both
155844 2022-01-01  155844     1   both

[155845 rows x 4 columns]
------------------- END VAL -----------

In [53]:
# TEST

check_correctness(dataset_arxiv, task_arxiv_paper_citation, rtgl_query, split="test")

TIMEDELTA: 182 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.15 seconds
------------------- START TEST -------------------
RelBench fkeys: {'Paper_ID': 'papers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'papers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2023-01-01       0     0   both
1      2023-01-01       1     0   both
2      2023-01-01       2     1   both
3      2023-01-01       3     0   both
4      2023-01-01       4     0   both
...           ...     ...   ...    ...
193691 2023-01-01  193691     0   both
193692 2023-01-01  193692     0   both
193693 2023-01-01  193693     0   both
193694 2023-01-01  193694     0   both
193695 2023-01-01  193695     0   both

[193696 rows x 4 columns]
------------------- END TEST ---------